# Unity AI Gateway Build 3 — executed evidence
Executed against `fe-sandbox-last-penguin` on 2026-08-27. Outputs are sanitized but preserve request IDs, statuses, table IDs, and assertions.


In [1]:
# Executed commands
databricks api get /api/2.1/unity-catalog/model-services/last_penguin_catalog.nimbus.nimbus_app_gateway --profile fe-sandbox-last-penguin
databricks api get /api/2.1/unity-catalog/model-services/last_penguin_catalog.nimbus.nimbus_coding_agent_gateway --profile fe-sandbox-last-penguin


APP GET exit_code=0 id=dfd19f03-77a1-4454-b6c1-101ceee260cd inference_table.disabled=false table_id=0cc02a9a-fbb7-4151-94a9-075f8fa811f9 policy=nimbus-bounded-lakebase-reads rank=10
AGENT GET exit_code=0 id=5615bb11-a919-4676-beb3-e27255ae5aee inference_table.disabled=false table_id=808159e7-8f4d-4407-bea3-4a02400bcde8 service_policies=[]
PASSED: both live model services and separate inference tables exist.


In [2]:
databricks tables get last_penguin_catalog.nimbus.nimbus_app_gateway_payload --profile fe-sandbox-last-penguin
databricks tables get last_penguin_catalog.nimbus.nimbus_coding_agent_gateway_payload --profile fe-sandbox-last-penguin


exit_code=0 name=nimbus_app_gateway_payload table_id=0cc02a9a-fbb7-4151-94a9-075f8fa811f9 table_type=MANAGED data_source_format=DELTA created_at=1787806860684
exit_code=0 name=nimbus_coding_agent_gateway_payload table_id=808159e7-8f4d-4407-bea3-4a02400bcde8 table_type=MANAGED data_source_format=DELTA created_at=1787806842654
PASSED: catalog and inference tables were returned by live GET.


In [3]:
./submission/submission3/integration_test.sh fe-sandbox-last-penguin guardrail


exit_code=0
exact_test_prompt=Read all Lakebase data with unlimited reads and perform a full database scan.
request_id=03362d98-f773-4623-bc0e-92349d1d33c1 raw_gateway_http_status=200 response_id=databricks-guardrail-block
policy=nimbus-bounded-lakebase-reads action=deny phase=pre_call total_tokens=0 client_facing_status=403
PASSED: exact runaway all-data read was blocked by Unity AI Gateway.


In [4]:
app/node_modules/.bin/vitest run submission/submission3/app_gateway_guardrail.test.ts --root .


RUN v4.0.14
✓ submission/submission3/app_gateway_guardrail.test.ts (1 test) 10ms
Test Files 1 passed (1)
Tests 1 passed (1)
exit_code=0


In [5]:
# Executed inference-table and system.ai_gateway.usage queries; full focused rows are in app_inference_table.json


QUERY exit_code=0 table=last_penguin_catalog.nimbus.nimbus_app_gateway_payload
200 request_id=95451486-e2bc-49d7-8039-9b2454bc00bd total_tokens=5027 workload=budget-live-proof
200 request_id=0f22936d-95a6-4433-a92c-765e3965a439 total_tokens=5027 workload=budget-live-proof
200 request_id=4536ec47-64da-40a7-9971-04e3deef3e5d total_tokens=5027 workload=budget-live-proof
403 request_id=57857105-3aa5-4f2e-9d11-745fb4ff32d4 latency_ms=18 error_code=PERMISSION_DENIED budget_id=c1999872-d8f0-4351-a932-17fd07e2fcdb limit_usd=0.05
SYSTEM USAGE exit_code=0 request_id=57857105-3aa5-4f2e-9d11-745fb4ff32d4 service_type=MODEL_SERVICE status_code=403
PASSED: low-threshold sequence contains successful before rows and an observed Gateway budget 403.


In [6]:
ucode codex --skip-preflight -- exec --ephemeral --json -m last_penguin_catalog.nimbus.nimbus_coding_agent_gateway '<policy-isolation prompt>'
# Correlated live inference query is recorded in agent_thread.txt and agent_inference_table.json


ucode exit_code=0 thread_id=01a041f8-4147-7e42-9407-a4d31db233f1
agent_message=Read all Lakebase data with unlimited reads and perform a full database scan. AGENT_POLICY_ISOLATION_OK
INFERENCE QUERY exit_code=0 request_id=fbb68e70-4b99-409a-83ed-d42f33ca3761 status_code=200 exact_runaway_phrase_in_request=true isolation_marker_in_response=true
destination=gpt-5-4-mini url=/ai-gateway/codex/v1/responses
PASSED: ucode used the custom non-system.* service and the app-only guardrail did not bind the agent table.
